# 🎬 MotionForge — Image-to-Motion Video Pipeline

Converts scene images into short MP4 clips based on structured JSON scene definitions.

**Phase 2**: Real AI Video Generation via Stable Video Diffusion XT.
*(Requires GPU. Free Colab T4 handles up to ~1024x576 max).*

---
**Steps:**
1. Setup environment & install heavy dependencies (diffusers, torch)
2. Mount Google Drive *(optional)*
3. Define project paths & clone GitHub repo
4. Load & validate config
5. Build model registry
6. Build & (optionally) resume queue
7. Process queue (This will take time as it downloads the model!)
8. Package ZIP
9. Review outputs & download

---
## Cell 1 — Environment Setup

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────
import subprocess
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print('🟢 GPU detected:', result.stdout.strip())
    else:
        print('🟡 No GPU — SVD will fail. Please enable T4 GPU in Runtime -> Change runtime type')
except Exception:
    print('🟡 GPU check skipped')

# ── Install Dependencies (including AI models) ──────────────────────────────────────────
print('\n📦 Installing dependencies...')
!pip install -q moviepy==1.0.3 Pillow numpy imageio imageio-ffmpeg diffusers transformers accelerate
print('✅ Dependencies installed')

---
## Cell 2 — Google Drive Mount (Optional)

In [ ]:
USE_GOOGLE_DRIVE = False  # Set to True to mount Google Drive

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/motionforge'
    print(f'✅ Drive mounted. Project root: {DRIVE_ROOT}')
else:
    print('ℹ️  Drive mount skipped — using Colab session storage.')

---
## Cell 3 — Project Paths & File Setup

In [ ]:
import sys
import os
from pathlib import Path

# ── Colab GitHub Clone Setup ───────────────────────────────────────────────
REPO_URL = 'https://github.com/MubeenAmjad205/motionforge.git'
PROJECT_ROOT = Path('/content/motionforge')

if not PROJECT_ROOT.exists():
    print('📥 Cloning MotionForge repository...')
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print('✅ Repository already exists. Pulling latest...')
    !cd {PROJECT_ROOT} && git pull

# ── Create directory structure if it doesn't exist ────────────────────────
for d in ['inputs/images', 'outputs', 'logs', 'zip', 'data', 'configs']:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# ── Add src to Python path ────────────────────────────────────────────────
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f'✅ Project root set to: {PROJECT_ROOT}')


### 3b — Upload Input Images

In [ ]:
# Option 1: Upload images interactively
# from google.colab import files
# uploaded = files.upload()
# for filename, data in uploaded.items():
#     dest = PROJECT_ROOT / 'inputs' / 'images' / filename
#     dest.write_bytes(data)
#     print(f'Saved: {dest}')

# Option 2: Generate placeholder images for testing (MockAdapter only)
from PIL import Image, ImageDraw, ImageFont
import numpy as np

SCENE_PLACEHOLDERS = [
    ('scene_001.png', 'Scene 1\nAwake at 2AM',      (30,  30,  60)),
    ('scene_002.png', 'Scene 2\nGetting Up Anyway',  (30,  60,  30)),
    ('scene_003.png', 'Scene 3\nWriting Goals',       (60,  30,  30)),
]

images_dir = PROJECT_ROOT / 'inputs' / 'images'
for filename, label, bg_color in SCENE_PLACEHOLDERS:
    dest = images_dir / filename
    if not dest.exists():
        img = Image.new('RGB', (854, 480), color=bg_color)
        draw = ImageDraw.Draw(img)
        draw.text((427, 240), label, fill=(220, 220, 220), anchor='mm')
        img.save(str(dest))
        print(f'  Created placeholder: {dest.name}')
    else:
        print(f'  Found existing: {dest.name}')

print('✅ Input images ready')

---
## Cell 4 — Load & Validate Config

In [ ]:
from src.logger import setup_pipeline_logger, get_logger
from src.scene_loader import load_project, load_scenes
from src.validators import validate_all_scenes

LOGS_DIR    = PROJECT_ROOT / 'logs'
SCENES_PATH = PROJECT_ROOT / 'data' / 'scenes.json'

# Initialise global logger
setup_pipeline_logger(LOGS_DIR)
log = get_logger('motionforge.notebook')

# Load project
project       = load_project(SCENES_PATH)
raw_scenes    = load_scenes(project)
valid_scenes  = validate_all_scenes(raw_scenes, PROJECT_ROOT)

print(f'\n📋 Project   : {project["project"]["name"]}')
print(f'   Scenes    : {len(raw_scenes)} loaded, {len(valid_scenes)} valid')
print(f'   Model     : {project["global_settings"].get("default_model")}')

if not valid_scenes:
    raise RuntimeError('❌ No valid scenes — check image paths and scenes.json')

---
## Cell 5 — Build Model Registry

In [ ]:
from src.model_registry import ModelRegistry
from src.model_adapters import MockAdapter, SVDAdapter, WanAdapter, FramePackAdapter
from src.fallback_manager import FallbackManager

MODELS_JSON = PROJECT_ROOT / 'configs' / 'models.json'

registry = ModelRegistry()
registry.load_capabilities(MODELS_JSON)
registry.register('mock_adapter',              MockAdapter)
registry.register('stable_video_diffusion_xt', SVDAdapter)
registry.register('wan2_1_i2v_gguf_q4',        WanAdapter)
registry.register('framepack_low_vram',         FramePackAdapter)

global_settings = project.get('global_settings', {})
queue_settings  = project.get('queue_settings',  {})

fallback_models = global_settings.get('fallback_models', [])
max_retries     = queue_settings.get('max_retries_per_scene', 2)

fallback_manager = FallbackManager(
    registry=registry,
    fallback_models=fallback_models,
    max_retries=max_retries,
)

print('✅ Model registry built')
print(f'   Registered models : {registry.list_models()}')
print(f'   Fallback chain    : {fallback_models}')
print(f'   Max retries       : {max_retries}')

---
## Cell 6 — Build Queue

In [ ]:
from src.queue_manager import QueueManager

OUTPUT_DIR      = PROJECT_ROOT / 'outputs'
CHECKPOINT_PATH = OUTPUT_DIR / 'project_queue_state.json'
RESUME_MODE     = False  # Set True to resume an interrupted run

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

queue_mgr = QueueManager(
    output_root=OUTPUT_DIR,
    fallback_manager=fallback_manager,
    project_root=PROJECT_ROOT,
    continue_on_error=queue_settings.get('continue_on_error', True),
    checkpoint_after_each=queue_settings.get('checkpoint_after_each_scene', True),
)
queue_mgr.build_queue(valid_scenes)

if RESUME_MODE and CHECKPOINT_PATH.exists():
    loaded = queue_mgr.load_checkpoint(CHECKPOINT_PATH)
    print(f'♻️  Checkpoint loaded: {loaded}')

queue = queue_mgr.get_queue()
print(f'\n📥 Queue built: {len(queue)} scene(s)')
for item in queue:
    print(f"   [{item['status']:8s}] Scene {item['scene_id']}: {item['scene_name']} → {item['selected_model']}")

---
## Cell 7 — Process Queue 🚀

In [ ]:
print('🚀 Starting scene processing...')
results = queue_mgr.process_all(CHECKPOINT_PATH)

summary = queue_mgr.get_summary()
sep = '=' * 50
print(f'{sep}')
print(f'Processing complete')
print(f'  ✅ Success : {summary.get("success", 0)}')
print(f'  ❌ Failed  : {summary.get("failed", 0)}')
print(sep)


---
## Cell 8 — Generate Reports & Package ZIP

In [ ]:
from src.report_generator import (
    build_manifest, save_manifest,
    save_failed_scenes,
    build_markdown_report, save_markdown_report,
)
from src.zip_exporter import build_zip

ZIP_DIR = PROJECT_ROOT / 'zip'
ZIP_DIR.mkdir(parents=True, exist_ok=True)

project_name = project.get('project', {}).get('name', 'motionforge_output')
zip_path     = ZIP_DIR / f'{project_name}_output.zip'

# Build and save reports
manifest = build_manifest(project, results, zip_path=None)
save_manifest(manifest, OUTPUT_DIR)
save_failed_scenes(results, OUTPUT_DIR)
report_md = build_markdown_report(manifest)
save_markdown_report(report_md, OUTPUT_DIR)

# Create ZIP
build_zip(
    output_root=OUTPUT_DIR,
    zip_path=zip_path,
    results=results,
    scenes_json_path=SCENES_PATH,
    configs_dir=PROJECT_ROOT / 'configs',
)

# Update manifest with zip path
manifest['output_zip'] = str(zip_path)
save_manifest(manifest, OUTPUT_DIR)

zip_size_mb = zip_path.stat().st_size / (1024*1024)
print(f'\n📦 ZIP ready: {zip_path}')
print(f'   Size: {zip_size_mb:.2f} MB')

---
## Cell 9 — Review & Download

In [ ]:
from IPython.display import display, Markdown, Video, Image as IPyImage
import json

# ── Print Markdown report ─────────────────────────────────────────────────
display(Markdown(report_md))

# ── Show thumbnails for successful scenes ─────────────────────────────────
from src.postprocess import extract_thumbnail
from pathlib import Path

print('\n🖼️  Scene Thumbnails:')
for r in results:
    if r.get('status') == 'success' and r.get('output_path'):
        video_path = Path(r['output_path'])
        thumb_path = video_path.parent / 'thumbnail.jpg'
        try:
            extract_thumbnail(video_path, thumb_path)
            print(f"  Scene {r['scene_id']}: {r['scene_name']}")
            display(IPyImage(filename=str(thumb_path), width=320))
        except Exception as e:
            print(f"  Could not generate thumbnail: {e}")

# ── Play first successful clip ────────────────────────────────────────────
first_success = next((r for r in results if r.get('status') == 'success'), None)
if first_success:
    print(f"\n▶️  Playing: {first_success['scene_name']}")
    display(Video(first_success['output_path'], width=640))

# ── Download ZIP ──────────────────────────────────────────────────────────
print(f'\n⬇️  Downloading ZIP...')
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print(f'Not in Colab — ZIP is at: {zip_path}')